In [ ]:
# 05_make_figures.ipynb
# Render the four data figures in grayscale with seaborn (Figures 3-6 of the
# manuscript): the scenario trade-off, the reliable-recourse comparison, the CDR
# sensitivity heatmap, and the Double Exclusion Index decomposition. Each is saved
# as PNG (600 dpi) and PDF into results/figures. Captions are omitted (they live in
# the manuscript). Figures 1-2 (concept and governance schematics) are prepared
# separately. No identifying paths are printed.

from pathlib import Path
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd(); ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
TAB = ROOT / "results" / "tables"; FIG = ROOT / "results" / "figures"; FIG.mkdir(parents=True, exist_ok=True)

sns.set_theme(context="paper", style="whitegrid", font="serif")
plt.rcParams.update({"figure.dpi": 150, "savefig.bbox": "tight", "axes.edgecolor": "#333333",
    "grid.color": "#DDDDDD", "font.size": 11})
GDARK, GMID, GLIGHT = "#444444", "#777777", "#BBBBBB"
def save(fig, name):
    fig.savefig(FIG / (name + ".png"), dpi=600); fig.savefig(FIG / (name + ".pdf")); plt.close(fig)

t1 = pd.read_csv(TAB / "table1_recourse_rates.csv")
t3 = pd.read_csv(TAB / "table3_fullpool_reliability.csv")
t3b = pd.read_csv(TAB / "table3b_advantaged_C_interval.csv").iloc[0]
t5 = pd.read_csv(TAB / "table5_dei.csv").set_index("group")
t6 = pd.read_csv(TAB / "table6_cdr_sensitivity.csv")

# Figure 3 - scenario trade-off (nominal vs reliable, by group).
scen = ["A\n(unconstrained)", "B\n(immutability)", "C\n(guardrail)"]
def val(g, metric):
    col = "RR_pct" if metric == "nominal" else "BR_RR_pct"
    return [t1.query("group==@g and scenario==@s")[col].iloc[0] for s in ["A", "B", "C"]]
fig, ax = plt.subplots(figsize=(7.1, 4.0)); x = np.arange(3); w = 0.2
for grp, metric, off, col, h in [("advantaged","nominal",-1.5,GLIGHT,""),("advantaged","reliable",-0.5,GDARK,""),
        ("disadvantaged","nominal",0.5,GLIGHT,"//"),("disadvantaged","reliable",1.5,GMID,"//")]:
    ax.bar(x + off*w, val(grp, metric), w, color=col, edgecolor="#222222", hatch=h,
        label=f"{grp.capitalize()} — {'Nominal' if metric=='nominal' else 'Reliable'} RR")
ax.set_xticks(x); ax.set_xticklabels(scen); ax.set_ylabel("Recourse rate (%)"); ax.set_ylim(0, 108)
ax.legend(fontsize=8, ncol=2, loc="upper center", framealpha=0.95); ax.grid(axis="x", visible=False)
save(fig, "fig1_scenario_tradeoff")

# Figure 4 - reliable recourse, advantaged vs disadvantaged, with 95% intervals.
est = [t3b["estimate_pct"], t3.query("scenario=='C'")["BR_RR_pct"].iloc[0]]
lo = [t3b["bootstrap_lo_pct"], 0.0]; hi = [t3b["bootstrap_hi_pct"], t3.query("scenario=='C'")["upper95_pct"].iloc[0]]
yerr = np.array([[est[i]-lo[i] for i in range(2)], [hi[i]-est[i] for i in range(2)]])
fig, ax = plt.subplots(figsize=(5.4, 4.0))
ax.bar([0, 1], est, 0.52, color=[GMID, GDARK], edgecolor="#222222", yerr=yerr, capsize=9, error_kw=dict(lw=1.3, ecolor="#222222"))
for i in range(2): ax.text(i, hi[i]+0.8, (f"{est[i]:.2f}%" if est[i] < 1 else f"{est[i]:.1f}%"), ha="center", fontsize=10)
ax.text(1, 3.3, f"95% upper\nbound {hi[1]:.2f}%", ha="center", fontsize=8, color="#333333")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Advantaged\n(n=200)", "Disadvantaged\n(full pool, n=785)"])
ax.set_ylabel("Borrower-Reliable RR (%), Scenario C"); ax.set_ylim(0, 23); ax.grid(axis="x", visible=False)
save(fig, "fig2_reliable_rr_groups")

# Figure 6 - CDR sensitivity heatmap (grayscale).
piv = t6.pivot(index="tenure", columns="age", values="CDR").reindex(index=["p33/p67","p25/p75","p20/p80"], columns=[">=50",">=60",">=70"])
fig, ax = plt.subplots(figsize=(5.6, 4.2))
sns.heatmap(piv, annot=np.array([[f"{v:.1f}x" for v in row] for row in piv.values]), fmt="", cmap="Greys", vmin=8, vmax=28,
    cbar_kws={"label": "CDR (advantaged / disadvantaged)"}, linewidths=0.6, linecolor="white", annot_kws={"fontsize": 11}, ax=ax)
ax.set_xlabel("Old-age-band threshold"); ax.set_ylabel("Tenure cutoff (short/long pct.)")
save(fig, "fig3_cdr_sensitivity")

# Figure 5 - Double Exclusion Index decomposition.
groups = ["Advantaged", "Disadvantaged"]
p_rej = [t5.loc["advantaged","P_reject"], t5.loc["disadvantaged","P_reject"]]
p_nr = [t5.loc["advantaged","P_no_recourse"], t5.loc["disadvantaged","P_no_recourse"]]
dei = [t5.loc["advantaged","DEI"], t5.loc["disadvantaged","DEI"]]
fig, ax = plt.subplots(figsize=(6.2, 4.0)); x = np.arange(2)
ax.bar(x-0.2, p_rej, 0.34, color=GLIGHT, edgecolor="#222222", label="P(rejection)")
ax.bar(x+0.2, p_nr, 0.34, color=GMID, edgecolor="#222222", hatch="//", label="P(no reliable recourse | rejected)")
ax.plot(x, dei, "o-", color="#111111", lw=1.6, ms=7, label="DEI = product (chain rule)")
for i in range(2): ax.text(x[i], dei[i]+0.03, f"DEI={dei[i]:.3f}", ha="center", fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(groups); ax.set_ylabel("Probability"); ax.set_ylim(0, 1.15)
ax.legend(fontsize=8, loc="lower right", framealpha=0.95); ax.grid(axis="x", visible=False)
save(fig, "fig4_dei_decomposition")

print("Notebook 05 complete. Figures 3-6 saved to results/figures as PNG (600 dpi) and PDF.")
